# Домашнее задание. Тема 5. Работа с файлами

Для каждого `user_id` из `visit_log.csv` определяем категорию покупки по файлу `purchase_log.txt` и записываем визиты с покупками в `funnel.csv`.

По условию `visit_log.csv` не помещается в память, поэтому он обрабатывается построчно. Покупки загружаем в словарь `user_id → категория`: это даёт поиск за O(1) на каждую строку визитов.

Замечания по данным:
- первая строка `purchase_log.txt` — это заголовок в формате JSON (`{"user_id": "user_id", "category": "category"}`), её пропускаем;
- если у пользователя несколько покупок, в словаре остаётся последняя категория из лога.

In [1]:
import json


def load_purchases(path):
    """Читает purchase_log построчно и возвращает словарь user_id -> category."""
    purchases = {}
    with open(path, encoding='utf-8') as file:
        next(file)  # строка-заголовок
        for line in file:
            record = json.loads(line)
            purchases[record['user_id']] = record['category']
    return purchases


def build_funnel(visit_path, purchases, funnel_path):
    """Построчно читает визиты и записывает в funnel только визиты с покупками."""
    written = 0
    with open(visit_path, encoding='utf-8') as visits, \
            open(funnel_path, 'w', encoding='utf-8') as funnel:
        header = next(visits).strip()
        funnel.write(f'{header},category\n')
        for line in visits:
            user_id, source = line.strip().split(',')
            category = purchases.get(user_id)
            if category is not None:
                funnel.write(f'{user_id},{source},{category}\n')
                written += 1
    return written

In [2]:
purchases = load_purchases('purchase_log.txt')
written = build_funnel('visit_log.csv', purchases, 'funnel.csv')
print(f'Покупателей в purchase_log: {len(purchases)}')
print(f'Записано визитов с покупками: {written}')

Покупателей в purchase_log: 99517
Записано визитов с покупками: 13798


Первые строки получившегося файла `funnel.csv`:

In [3]:
with open('funnel.csv', encoding='utf-8') as funnel:
    for _ in range(3):
        print(next(funnel).strip())

user_id,source,category
1840e0b9d4,other,Продукты
4e4f90fcfb,context,Электроника
